# Model Based Grading

Time to replace the hardcoded `score = 10` with a real grader. A grader takes model output and returns a measurable signal — usually a number **1–10**.

**Three kinds of graders:**

| Grader | How it works | Good for |
|--------|--------------|----------|
| **Code** | Programmatic checks | Length, banned words, JSON/Python/regex syntax, readability |
| **Model** | A second Claude call scores the output | Response quality, instruction-following, completeness, helpfulness, safety |
| **Human** | People review manually | Nuanced quality, depth, relevance — but slow & tedious |

**Our evaluation criteria** (for the AWS code-gen prompt):
- **Format** — return *only* Python / JSON / Regex, no explanation → *code grader* (next lesson)
- **Valid syntax** — output should parse → *code grader* (next lesson)
- **Task following** — accurately addresses the request → *model grader* (this lesson)

This notebook implements the **model grader**: feed the task + solution to Claude and ask for a structured JSON verdict.

> **Adaptations:**
> - Runs on **Haiku 4.5** — the grader uses assistant prefilling (` ```json `), which 400s on flagship models.
> - **Grader prompt matches the course's *downloadable* notebook, not the lesson-page snippet.** The lesson snippet is a *plain* string whose `{task}` / `{solution}` placeholders never get filled — the grader would literally see `{task}`. The real downloadable uses an **f-string** with `<task>`/`<solution>` XML tags and an example response shape; we use that fuller, correct version.

## Setup — helpers + `run_prompt`

In [4]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

import json
from statistics import mean
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5-20251001"   # grader uses prefill; also fast/cheap for eval loops


def get_text(message):
    for block in message.content:
        if block.type == "text":
            return block.text
    return ""


def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }

    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return get_text(message)


def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""

    messages = []
    add_user_message(messages, prompt)
    return chat(messages)


print(f"Ready. Model: {model}")

Ready. Model: claude-haiku-4-5-20251001


## The model grader

The key trick: ask for **strengths, weaknesses, and reasoning** *alongside* the score. Without that context, models default to middling ~6 scores. Forcing it to justify the verdict produces a more discriminating number.

We reuse the prefill + stop-sequence pattern to get raw JSON back.

In [3]:
def grade_by_model(test_case, output):
    # f-string so {task}/{output} interpolate — matches the course's downloadable notebook
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
"""

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")   # prefill: open the JSON block

    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

## Wire the grader into the pipeline

`run_test_case` now calls the grader and pulls out the `score` and `reasoning`.

In [5]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

## `run_eval` with an average score

The average across all test cases is the single objective number we track as we iterate on the prompt.

In [6]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

## Run it

Now each case costs *two* calls (solve + grade), so this is slower than the last lesson. The baseline prompt is still bare, so expect the model grader to ding it on the verbose, unformatted output.

In [7]:
DATASET_PATH = "01-building-with-the-claude-api/02-prompt-evaluation/dataset.json"

with open(DATASET_PATH, "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)
print(json.dumps(results, indent=2))

Average score: 6.333333333333333
[
  {
    "output": "# AWS S3 Region Extraction Function\n\nHere's a comprehensive solution with multiple approaches:\n\n## Basic Solution\n\n```python\nimport re\nfrom urllib.parse import urlparse\n\ndef extract_s3_region(bucket_url: str) -> str:\n    \"\"\"\n    Extract AWS region from an S3 bucket URL.\n    \n    Args:\n        bucket_url: S3 bucket URL (e.g., 's3.us-west-2.amazonaws.com' or \n                   'https://bucket-name.s3.us-west-2.amazonaws.com')\n    \n    Returns:\n        Region code (e.g., 'us-west-2') or None if region not found\n    \n    Raises:\n        ValueError: If the URL format is invalid\n    \"\"\"\n    if not bucket_url:\n        raise ValueError(\"Bucket URL cannot be empty\")\n    \n    # Remove protocol if present\n    if \"://\" in bucket_url:\n        bucket_url = bucket_url.split(\"://\", 1)[1]\n    \n    # Pattern to match region in S3 URLs\n    # Matches: s3.region.amazonaws.com or bucket.s3.region.amazonaws.com

## Takeaway

Model graders are flexible but a little **capricious** — the same output can score slightly differently run to run. They still give a consistent-enough baseline to measure prompt changes against. The two criteria a model grader is *bad* at — exact format and valid syntax — are exactly what **code-based grading** (next lesson) nails deterministically.